In [1]:
import csv
import itertools
import plotly.graph_objects as go
import plotly.io as pio
from collections import Counter
from cohort_const import CohortColorPallet

cohort_colors = CohortColorPallet()

In [2]:
class SankeyNode:
    def __init__(self, type, label) -> None:
        self.type = type
        self.label = label
        self.prior_mdx_order = [
            "Type I & 15q26",
            "Type I",
            "Deletion (Unspecified)",
            "Type II",
            "Deletion & Translocation",
            "IC deletion",
            "Unknown",
            "UPD",
            "Mosaic Epimutation",
            "Epimutation",
        ]
        self.gs_mdx_order = [
            "Classic Type I",
            "Complex Type I",
            "Classic Type II",
            "Complex Type II",
            "Atypical deletion",
            "Atypical deletion*",
            "IC deletion",
            "Isodisomy",
            "Segmental Isodisomy",
            "Heterodisomy or Epimutation",
            "Atypical UPD",
        ]
        self.gs_mdx_cat_order = [
            "Deletion",
            "Deletion*",
            "UPD/ICD",
        ]

    def getX(self):
        match self.type:
            case "Prior":
                return 0.01
            case "GS":
                return 0.5
            case _:
                return 0.99

    def getCatColumnOrder(self):
        match self.type:
            case "Prior":
                return self.prior_mdx_order
            case "GS":
                return self.gs_mdx_order
            case _:
                return self.gs_mdx_cat_order

    def getRGBA(self, alpha=1.0):
        return cohort_colors.getCategoryColorRGBA(self.label, alpha)

    def __eq__(self, other):
        if not isinstance(other, SankeyNode):
            return NotImplemented
        return self.type == other.type and self.label == other.label

    def __hash__(self):
        return hash((self.type, self.label))

    def __str__(self) -> str:
        return f"SankeyNode(type = {self.type}, label = {self.label})"


def get_link_color(src: SankeyNode, target: SankeyNode):
    if target.type == "mdx_cat":
        return target.getRGBA(alpha=0.55)
    else:
        return src.getRGBA(alpha=0.55)


def inc_dict_cntr(cntr_dict: dict, key: str):
    if key not in cntr_dict:
        cntr_dict[key] = 0
    cntr_dict[key] += 1


def get_node_y(node: SankeyNode, cat_cntr_dict: dict) -> float:
    cat_order = node.getCatColumnOrder()
    # first node in each column should have a y-value of 0.001
    node_order_idx = cat_order.index(node.label)
    if node_order_idx == 0:
        return 0.001

    # get all node labels between first node and this node
    btw_node_labels = cat_order[1:node_order_idx]
    btw_num_parts = sum(cat_cntr_dict[node_label] for node_label in btw_node_labels)
    return float(btw_num_parts) * 0.02 + float(cat_cntr_dict[node.label]) * 0.01


links = []
prior_mdx_cnts = {}
gs_mdx_cnts = {}
mdx_cat_cnts = {}
with open(
    "../data/raw/pws-reported-primary-findings-variants.csv", "r", encoding="utf-8"
) as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        inc_dict_cntr(prior_mdx_cnts, row["SOURCE_MDX"])
        prior_mdx = SankeyNode("Prior", row["SOURCE_MDX"])

        inc_dict_cntr(gs_mdx_cnts, row["REVISED_GS_SUB_CAT"])
        gs_mdx = SankeyNode("GS", row["REVISED_GS_SUB_CAT"])

        inc_dict_cntr(mdx_cat_cnts, row["GENERIC_REVISED_GS_SUB_CAT"])
        mdx_cat = SankeyNode("mdx_cat", row["GENERIC_REVISED_GS_SUB_CAT"])

        links.append((prior_mdx, gs_mdx))
        links.append((gs_mdx, mdx_cat))

link_counts = Counter(links)
# create unique list of nodes
nodes = list({node for node in itertools.chain(*links)})

# set up data structures for the diagram
labels = []
node_colors = []
node_x = []
node_y = []
for node in nodes:
    node_colors.append(node.getRGBA())
    node_x.append(node.getX())
    match node.type:
        case "Prior":
            labels.append(f"({prior_mdx_cnts[node.label]}) {node.label}")
            node_y.append(get_node_y(node, prior_mdx_cnts))
        case "GS":
            labels.append(f"({gs_mdx_cnts[node.label]}) {node.label}")
            node_y.append(get_node_y(node, gs_mdx_cnts))
        case _:
            labels.append(f"({mdx_cat_cnts[node.label]}) {node.label}")
            node_y.append(get_node_y(node, mdx_cat_cnts))

sources = []
targets = []
values = []
link_colors = []
for link, value in link_counts.items():
    sources.append(nodes.index(link[0]))
    targets.append(nodes.index(link[1]))
    values.append(value)
    link_colors.append(link[0].getRGBA(alpha=0.55))

fig = go.Figure(
    data=[
        go.Sankey(
            arrangement="snap",
            node={
                "label": labels,
                "x": node_x,
                "y": node_y,
                "pad": 5,  # pad nodes by 5 pixels
                "color": node_colors,
            },
            link=dict(source=sources, target=targets, value=values, color=link_colors),
        )
    ]
)

fig.update_layout(
    font_size=14,
    autosize=False,
    height=700,
    width=1000,
)
fig.show()
# export to file
fig.write_image("../data/processed/pws-mdx-sankey.png", height=700, width=1000, scale=2)